# 01 — AutoML (D1 rework, Pair B)

Marker's D1 critique on AutoML was 20/50: "just tried one optimiser — just to tick the box?". This notebook is the artifact-only view of the rework. Three things to surface:

1. **5-method HPO comparison** — Grid / Random / TPE / Hyperband / BOHB on the same 240/60 split, ranked by holdout NDCG@5 (`reports/sampler_comparison.csv`).
2. **Search-space ablation** — drop each tuned dim back to its default, re-evaluate on the holdout, report the delta (`reports/search_space_ablation.csv`).
3. **Blessed configuration** — `configs/winning_runcard.yaml`, schema v3, BOHB winner over the expanded 7-D space (B2 added BM25 k1/b).

Run all 5 methods + write the runcard end-to-end with:
```
python -m csai415.hpo_methods
```
Run the ablation against the blessed winner with:
```
python -m csai415.ablation
```
This notebook reads the artifacts those commands produce. Re-running it doesn't re-train anything.

In [ ]:
from pathlib import Path
import ast
import pandas as pd
import yaml
from IPython.display import Markdown, display

## 1 — 5-method HPO comparison

Each method ran 80 trials (Grid: 48 cells) on the same 240-query tune set, then the winner was evaluated once on the 60-query holdout. The original D1 was a single-TPE study; this is the multi-optimiser answer to the marker's critique.

In [ ]:
comparison = pd.read_csv('../reports/sampler_comparison.csv')
comparison['best_params'] = comparison['best_params'].apply(ast.literal_eval)
comparison.sort_values('test_score', ascending=False).reset_index(drop=True)

Note the mixed search-space dimensionality:
* Grid / Random / TPE / Hyperband searched the canonical 5-D space (metric, svd_dim, normalize, hybrid_weight, candidate_k).
* BOHB was re-tuned over the expanded 7-D space (adds BM25 k1/b) so the top-level `best_params` covers every dim the ablation tests.

Grid is the marginal holdout leader but Grid / BOHB / TPE are within paired-bootstrap noise (paired bootstrap B=5000 over the original 5-D points: P(tie | Grid vs BOHB) = 1.0). BOHB stays blessed because (a) the multi-fidelity ladder finds the same optimum at lower wall-clock and (b) it carries the tuned BM25 params the ablation needs.

## 2 — Search-space ablation (B2)

Hold the 7-D BOHB winner fixed; drop one dim at a time back to its `RetrieverConfig` default; re-evaluate on the same 60 holdout queries. `delta_ndcg5` is the change in NDCG@5 vs the full winner:

* large negative delta → the optimiser's tuned value mattered,
* delta ~ 0 → the default was already the winner's choice (or close enough),
* positive delta → overfit signal: the tuned value is worse than the default.

In [ ]:
ablation = pd.read_csv('../reports/search_space_ablation.csv')
ablation

Honest read: `hybrid_weight` is the only dim AutoML earned by a wide margin (-0.026); `bm25_b` genuinely helps (-0.004); `bm25_k1`'s tuned value (2.92) mildly OVERFITS to the tune set (+0.0018 when dropped back to 1.5). The remaining four dims sit at noise.

That's a stronger argument than "all 7 dims helped" — the marker can verify the claim against the table.

## 3 — Blessed runcard

Schema v3 carries `blessed_method`, the inline 5-method comparison, the comparison-CSV path, and the v2-shaped metrics block so downstream readers (Pair C, the report) don't need code changes.

In [ ]:
runcard = yaml.safe_load(Path('../configs/winning_runcard.yaml').read_text(encoding='utf-8'))
print(f"schema_version : {runcard['schema_version']}")
print(f"blessed_method : {runcard['automl']['blessed_method']}")
print(f"sampler        : {runcard['automl']['sampler']['class']}")
print(f"pruner         : {runcard['automl']['pruner']['class']}")
print(f"trials (total) : {runcard['automl']['n_trials']}")
print(f"best tune NDCG : {runcard['automl']['best_value_ndcg5_tune']:.4f}")
print('best_params    :')
for k, v in runcard['automl']['best_params'].items():
    print(f'  {k:<14}: {v}')
print('holdout metrics:')
for k, v in runcard['metrics']['winner_holdout'].items():
    print(f'  {k:<14}: {v:.4f}')

In [ ]:
display(Markdown(runcard['notes']))